In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from pathlib import Path

ROOT = Path().resolve().parent
DATA_RAW = ROOT / "data" / "raw"
DATA_PROCESSED = ROOT / "data" / "processed"

df = pd.read_csv(DATA_RAW / "ai4i2020.csv")
print("Shape:", df.shape)
df.head()

Shape: (10000, 14)


,UDI,Product ID,Type,Air temperature [K],Process temperature [K],Rotational speed [rpm],Torque [Nm],Tool wear [min],Machine failure,TWF,HDF,PWF,OSF,RNF
0,1,M14860,M,298.1,308.6,1551,42.8,0,0,0,0,0,0,0
1,2,L47181,L,298.2,308.7,1408,46.3,3,0,0,0,0,0,0
2,3,L47182,L,298.1,308.5,1498,49.4,5,0,0,0,0,0,0
3,4,L47183,L,298.2,308.6,1433,39.5,7,0,0,0,0,0,0
4,5,L47184,L,298.2,308.7,1408,40.0,9,0,0,0,0,0,0


In [ ]:
# Feature 1: Temperature difference
# Process temperature is always higher than air temperature
# When this gap grows abnormally, it signals the machine is generating excess heat
df['temp_difference'] = df['Process temperature [K]'] - df['Air temperature [K]']

print("temp_difference stats:")
print(df['temp_difference'].describe().round(2))

temp_difference stats:
count    10000.0
mean        10.0
std          1.0
min          7.6
25%          9.3
50%          9.8
75%         11.0
max         12.1
Name: temp_difference, dtype: float64


In [3]:
print("Average temp_difference by failure:")
print(df.groupby('Machine failure')['temp_difference'].mean().round(3))

Average temp_difference by failure:
Machine failure
0    10.022
1     9.404
Name: temp_difference, dtype: float64


In [ ]:
# Power.
# Feature 2: Power = Torque × Rotational speed
# Captures the combined mechanical stress on the machine
# We divide by 9550 to convert to kilowatts (standard engineering conversion)
df['power'] = (df['Torque [Nm]'] * df['Rotational speed [rpm]']) / 9550

print("power stats:")
print(df['power'].describe().round(2))

power stats:
count    10000.00
mean         6.28
std          1.07
min          1.15
25%          5.56
50%          6.27
75%          7.00
max         10.47
Name: power, dtype: float64


In [5]:
print("Average power by failure:")
print(df.groupby('Machine failure')['power'].mean().round(3))

Average power by failure:
Machine failure
0    6.244
1    7.282
Name: power, dtype: float64


In [ ]:
# Feature 3: Tool wear ratio
# How close is the tool to its maximum observed wear limit
MAX_WEAR = 253

df['tool_wear_ratio'] = df['Tool wear [min]'] / MAX_WEAR

print("tool_wear_ratio stats:")
print(df['tool_wear_ratio'].describe().round(3))

print("\nAverage tool_wear_ratio by failure:")
print(df.groupby('Machine failure')['tool_wear_ratio'].mean().round(3))

tool_wear_ratio stats:
count    10000.000
mean         0.427
std          0.252
min          0.000
25%          0.209
50%          0.427
75%          0.640
max          1.000
Name: tool_wear_ratio, dtype: float64

Average tool_wear_ratio by failure:
Machine failure
0    0.422
1    0.568
Name: tool_wear_ratio, dtype: float64


In [10]:
# Feature 4: Type-encoded distribution: Ordinal encoding of product type
# There is a natural order: Low < Medium < High quality
# So we encode as numbers that preserve that order
type_mapping = {'L': 0, 'M': 1, 'H': 2}
df['type_encoded'] = df['Type'].map(type_mapping)

print("type_encoded distribution:")
print(df.groupby('Type')['type_encoded'].first())

type_encoded distribution:
Type
H    2
L    0
M    1
Name: type_encoded, dtype: int64


In [11]:
engineered_features = ['temp_difference', 'power', 'tool_wear_ratio', 'type_encoded']

print("New features summary:")
print(df[engineered_features].describe().round(3))

New features summary:
       temp_difference      power  tool_wear_ratio  type_encoded
count        10000.000  10000.000        10000.000     10000.000
mean            10.001      6.279            0.427         0.500
std              1.001      1.067            0.252         0.671
min              7.600      1.148            0.000         0.000
25%              9.300      5.561            0.209         0.000
50%              9.800      6.271            0.427         0.000
75%             11.000      7.002            0.640         1.000
max             12.100     10.469            1.000         2.000


In [12]:
# Select final columns for modelling
final_columns = [
    'type_encoded',
    'Air temperature [K]',
    'Process temperature [K]',
    'Rotational speed [rpm]',
    'Torque [Nm]',
    'Tool wear [min]',
    'temp_difference',
    'power',
    'tool_wear_ratio',
    'Machine failure',
    'TWF', 'HDF', 'PWF', 'OSF', 'RNF'
]

df_processed = df[final_columns]
df_processed.to_csv(DATA_PROCESSED / "ai4i_processed.csv", index=False)
print("Saved processed dataset with shape:", df_processed.shape)

Saved processed dataset with shape: (10000, 15)
